# Stroke Prediction: Five Classifiers (ML Assignment 2)

**How to use this notebook:** run the cells **one by one, from top to bottom** (Shift + Enter). Every code cell has a note above it saying what it does and what output you should see. Do not skip cells: later cells use variables created by earlier ones.

| Step | Cells | What happens |
|---|---|---|
| 1 | 1-3 | Import libraries, load the dataset, look at it |
| 2 | 4-6 | 80/20 train-test split, then balance the training data |
| 3 | 7-8 | Build the preprocessing pipeline and the evaluation function |
| 4 | 9-13 | Train and evaluate the 5 models, one cell each |
| 5 | 14-16 | Comparison table, confusion matrices, classification reports |
| 6 | 17 | Save the models and the test file used by the Streamlit app |

After the last cell, run the app from a terminal: `streamlit run app.py`

**Folder layout needed before you start** (notebook sits in the project root):
```
stroke-ml-app/
  stroke_ml_assignment.ipynb   <- this notebook
  app.py
  data/healthcare-dataset-stroke-data.csv
```

## Step 1: Setup and data

### Cell 1: Import libraries
Expected output: `Libraries loaded`

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef,
                             confusion_matrix, classification_report)

RANDOM_STATE = 42          # fixed seed so results are reproducible
os.makedirs("model", exist_ok=True)
print("Libraries loaded")

### Cell 2: Load the dataset
Reads the Kaggle stroke dataset. Expected output: shape `(5110, 12)` and the first 5 rows.

In [ ]:
DATA_PATH = "data/healthcare-dataset-stroke-data.csv"
if not os.path.exists(DATA_PATH):                      # fall back to same folder
    DATA_PATH = "healthcare-dataset-stroke-data.csv"

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())

### Cell 3: Understand the data
Checks column types, missing values and the class balance.

Expected: `bmi` has **201** missing values, and the target `stroke` has **4861** zeros and only **249** ones (about 4.9%). That heavy imbalance is the main challenge of this dataset, and it is why we balance the training data later.

In [ ]:
print("Column types:")
print(df.dtypes, "\n")

print("Missing values per column:")
print(df.isna().sum(), "\n")

print("Target distribution (stroke):")
print(df["stroke"].value_counts())
print((df["stroke"].value_counts(normalize=True) * 100).round(2).astype(str) + " %")

ax = df["stroke"].value_counts().plot(kind="bar", color=["steelblue", "indianred"])
ax.set_title("Class distribution: 0 = no stroke, 1 = stroke")
ax.set_xlabel("stroke")
ax.set_ylabel("patients")
plt.xticks(rotation=0)
plt.show()

## Step 2: Train/test split and balancing

### Cell 4: 80/20 split
`stratify` keeps the same stroke percentage in both parts. The test part is saved as `test_data.csv`; this is the file you upload in the Streamlit app.

Expected: train **4088** rows (199 stroke), test **1022** rows (50 stroke).

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=RANDOM_STATE, stratify=df["stroke"]
)

train_df.to_csv("model/train_data.csv", index=False)
test_df.to_csv("test_data.csv", index=False)

print("Train rows:", len(train_df), "| stroke cases:", train_df["stroke"].sum())
print("Test rows :", len(test_df),  "| stroke cases:", test_df["stroke"].sum())
print("Saved: model/train_data.csv and test_data.csv")

### Cell 5: Choose features and target
`id` is only a row label, so it is dropped. The other 10 columns are the input features.

Expected: `X_train (4088, 10)  X_test (1022, 10)`

In [ ]:
TARGET = "stroke"
NUMERIC_COLS = ["age", "avg_glucose_level", "bmi"]
BINARY_COLS = ["hypertension", "heart_disease"]
CATEGORICAL_COLS = ["gender", "ever_married", "work_type", "Residence_type", "smoking_status"]
FEATURE_COLS = NUMERIC_COLS + BINARY_COLS + CATEGORICAL_COLS

X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET]
X_test,  y_test  = test_df[FEATURE_COLS],  test_df[TARGET]
print("X_train", X_train.shape, " X_test", X_test.shape)

### Cell 6: Remove data skewness (oversampling the training set only)
The syllabus lists *data skewness removal (sampling)*. Stroke cases are randomly duplicated in the **training** data until both classes are equal. The test data is left untouched so the final scores reflect real-world class proportions.

Expected: before `{0: 3889, 1: 199}`, after `{0: 3889, 1: 3889}`

In [ ]:
print("Before:", y_train.value_counts().to_dict())

minority = X_train[y_train == 1]
extra = resample(minority, replace=True,
                 n_samples=(y_train == 0).sum() - (y_train == 1).sum(),
                 random_state=RANDOM_STATE)

X_bal = pd.concat([X_train, extra])
y_bal = pd.concat([y_train, pd.Series(1, index=extra.index)])

# shuffle so the duplicated rows are mixed in
order = np.random.RandomState(RANDOM_STATE).permutation(len(X_bal))
X_bal = X_bal.iloc[order].reset_index(drop=True)
y_bal = y_bal.iloc[order].reset_index(drop=True)

print("After :", y_bal.value_counts().to_dict())

## Step 3: Pipeline and evaluation function

### Cell 7: Preprocessing pipeline
- numeric columns: missing values filled with the median, then scaled (mean 0, std 1)
- categorical columns: one-hot encoded
- binary columns: passed through

Every model is wrapped as `preprocessing + classifier`, so the preprocessing is learned from training data only and applied identically to test data.

Expected output: `Pipeline builder ready`

In [ ]:
def make_pipeline(classifier):
    numeric = Pipeline([("impute", SimpleImputer(strategy="median")),
                        ("scale", StandardScaler())])
    prep = ColumnTransformer([
        ("num", numeric, NUMERIC_COLS),
        ("bin", "passthrough", BINARY_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLS),
    ])
    return Pipeline([("prep", prep), ("clf", classifier)])

print("Pipeline builder ready")

### Cell 8: Evaluation function
Computes the six metrics the assignment asks for (Accuracy, AUC, Precision, Recall, F1, MCC) on the **test** data. `results` collects one row per model; `trained` keeps the fitted models.

Expected output: `Evaluation function ready`

In [ ]:
results = {}
trained = {}

def evaluate(name, pipe):
    pred = pipe.predict(X_test)
    # AUC needs a continuous score: probability if available, otherwise decision function
    if hasattr(pipe, "predict_proba"):
        score = pipe.predict_proba(X_test)[:, 1]
    else:
        score = pipe.decision_function(X_test)
    results[name] = {
        "Accuracy":  accuracy_score(y_test, pred),
        "AUC":       roc_auc_score(y_test, score),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall":    recall_score(y_test, pred, zero_division=0),
        "F1":        f1_score(y_test, pred, zero_division=0),
        "MCC":       matthews_corrcoef(y_test, pred),
    }
    trained[name] = pipe
    for metric, value in results[name].items():
        print(f"{metric:10s}: {value:.4f}")

print("Evaluation function ready")

## Step 4: Train the five models
Each cell trains one model on the balanced training data and prints its test-set metrics. The expected values are given so you can check your run against mine.

### Cell 9: Model 1, Logistic Regression
Expected: Accuracy 0.7446, AUC 0.8444, Precision 0.1375, Recall 0.8000, F1 0.2346, MCC 0.2590

In [ ]:
model = make_pipeline(LogisticRegression(max_iter=1000))
model.fit(X_bal, y_bal)
evaluate("Logistic Regression", model)

### Cell 10: Model 2, Decision Tree
Uses the **entropy** criterion (information gain, as in the syllabus) and a depth limit of 5 to avoid overfitting.

Expected: Accuracy 0.7221, AUC 0.7773, Precision 0.1201, Recall 0.7400, F1 0.2067, MCC 0.2168

In [ ]:
model = make_pipeline(DecisionTreeClassifier(criterion="entropy", max_depth=5,
                                             random_state=RANDOM_STATE))
model.fit(X_bal, y_bal)
evaluate("Decision Tree", model)

### Cell 11: Model 3, k-Nearest Neighbors
Expected: Accuracy 0.8483, AUC 0.5998, Precision 0.0732, Recall 0.1800, F1 0.1040, MCC 0.0416

Note the high accuracy but very low recall: kNN misses most stroke cases.

In [ ]:
model = make_pipeline(KNeighborsClassifier(n_neighbors=5))
model.fit(X_bal, y_bal)
evaluate("kNN", model)

### Cell 12: Model 4, Gaussian Naive Bayes
Expected: Accuracy 0.2299, AUC 0.7850, Precision 0.0587, Recall 0.9800, F1 0.1107, MCC 0.0956

Note the opposite pattern: it catches almost every stroke but flags most healthy patients too.

In [ ]:
model = make_pipeline(GaussianNB())
model.fit(X_bal, y_bal)
evaluate("Naive Bayes", model)

### Cell 13: Model 5, Support Vector Machine
RBF kernel. This is the slowest of the five to train, but it should still finish within a few seconds on a normal lab machine.

Expected: Accuracy 0.7681, AUC 0.7815, Precision 0.1184, Recall 0.5800, F1 0.1966, MCC 0.1808

In [ ]:
model = make_pipeline(SVC(kernel="rbf", random_state=RANDOM_STATE))
model.fit(X_bal, y_bal)
evaluate("Support Vector Machine", model)

## Step 5: Compare the models

### Cell 14: Comparison table
This is the table that goes into the README (section d). The second part prints the best model for each metric.

In [ ]:
comparison = pd.DataFrame(results).T.round(4)
comparison.index.name = "ML Model Name"
display(comparison)

print("\nBest model per metric:")
for metric in comparison.columns:
    print(f"{metric:10s}: {comparison[metric].idxmax()}  ({comparison[metric].max():.4f})")

### Cell 15: Confusion matrices
Each box shows how many test patients were classified correctly or wrongly. The bottom-left box is **missed strokes** (false negatives); the top-right is **false alarms** (false positives).

In [ ]:
labels = ["No stroke", "Stroke"]
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, (name, pipe) in zip(axes, trained.items()):
    cm = confusion_matrix(y_test, pipe.predict(X_test), labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

### Cell 16: Classification reports
Precision, recall and F1 for each class, per model.

In [ ]:
for name, pipe in trained.items():
    print("=" * 60)
    print(name)
    print(classification_report(y_test, pipe.predict(X_test),
                                target_names=labels, zero_division=0))

## Step 6: Save everything for the Streamlit app

### Cell 17: Save models and metrics
Writes the five fitted models into the `model/` folder (the names below are the ones `app.py` looks for) and saves the metrics table.

Expected: five `saved:` lines, then a list of files in `model/`.

In [ ]:
files = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "kNN": "knn.joblib",
    "Naive Bayes": "naive_bayes.joblib",
    "Support Vector Machine": "svm.joblib",
}
for name, filename in files.items():
    joblib.dump(trained[name], os.path.join("model", filename))
    print("saved:", filename)

comparison.to_csv("model/metrics_summary.csv")
print("\nFiles in model/:", sorted(os.listdir("model")))

## What to do next

1. Open a terminal in the project folder and install the libraries: `pip install -r requirements.txt`
2. Start the app: `streamlit run app.py`. In the sidebar, upload `test_data.csv` (created in Cell 4), pick a model, and check that the metrics match the table from Cell 14.
3. Paste the Cell 14 table into the README (section d) if your numbers differ from the README.
4. Push the folder to GitHub, then deploy on Streamlit Community Cloud with `app.py`.